# Mini-YOLO Fine-Tuning on COCO Dataset

In the last notebook, we have trained a **Mini-YOLO** model from scratch to detect synthetic shapes like triangles or circles. Now let's challenge our model and see how it performs on real images! 
We will fine-tune the Mini-YOLO model on a subset of the COCO dataset to detect persons in real images.

**Steps we will complete:**
- Download and extract COCO images and labels
- Parse labels for selected class
- Encode YOLO targets (cell-relative)
- Fine-tune Mini-YOLO
- Visualize predictions (with optional NMS)


In [ ]:

import os, zipfile, urllib.request, random
from glob import glob

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import cv2
import matplotlib.pyplot as plt

import copy
import math

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True


## 1) Download COCO val2017 images and annotations

First, we need to get the data. To keep the dataset small, we use **val2017** (5,000 images). So we download:
- `val2017.zip` (images)
- `annotations_trainval2017.zip` (contains `instances_val2017.json` for boxes and masks)

In the classroom workspace, the data has already been downloaded for you. In case you are working locally on your computer, the download will happen automatically - potentially adapt the `DATA_ROOT` path to your system.

In [ ]:
from pathlib import Path
import urllib.request
import zipfile

# TODO: potentially adapt this path in case you are working on your local computer
DATA_ROOT = Path("/data") 
DATA_ROOT.mkdir(exist_ok=True)

# COCO directory layout after extraction:
# /data/coco/
#   val2017/                      (images)
#   annotations/instances_val2017.json  (boxes + masks)
COCO_ROOT = DATA_ROOT / "coco"
COCO_ROOT.mkdir(exist_ok=True)

COCO_IMAGES_DIR = COCO_ROOT / "val2017"
COCO_ANN_DIR    = COCO_ROOT / "annotations"
COCO_ANN_FILE   = COCO_ANN_DIR / "instances_val2017.json"

VAL_ZIP = COCO_ROOT / "val2017.zip"
ANN_ZIP = COCO_ROOT / "annotations_trainval2017.zip"

VAL_URL = "http://images.cocodataset.org/zips/val2017.zip"
ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"


In [ ]:
# Check if COCO val2017 images and annotations are already present
images_ok = COCO_IMAGES_DIR.exists() and len(list(COCO_IMAGES_DIR.glob("*.jpg"))) > 0
ann_ok    = COCO_ANN_FILE.exists()

if images_ok and ann_ok:
    print("COCO val2017 images + annotations already present — skipping download.")
else:
    print("COCO not found locally yet. Will download missing parts.")


In [ ]:
# Download + extract val2017 images
if not images_ok:
    if not VAL_ZIP.exists():
        print("Downloading COCO val2017 images ... (this can take a while, ~1 GB)")
        urllib.request.urlretrieve(VAL_URL, VAL_ZIP.as_posix())
        print("Images downloaded:", VAL_ZIP)

    print("Extracting COCO val2017 images ...")
    with zipfile.ZipFile(VAL_ZIP, "r") as zf:
        zf.extractall(COCO_ROOT)
    print("Images extracted to:", COCO_IMAGES_DIR)

# Download + extract annotations (contains instances_val2017.json)
if not ann_ok:
    if not ANN_ZIP.exists():
        print("Downloading COCO annotations ... (~241 MB)")
        urllib.request.urlretrieve(ANN_URL, ANN_ZIP.as_posix())
        print("Annotations downloaded:", ANN_ZIP)

    print("Extracting COCO annotations ...")
    with zipfile.ZipFile(ANN_ZIP, "r") as zf:
        zf.extractall(COCO_ROOT)
    print("Annotations extracted to:", COCO_ANN_DIR)


In [ ]:
# Final checks
assert COCO_IMAGES_DIR.exists(), f"Image directory not found: {COCO_IMAGES_DIR}"
assert COCO_ANN_FILE.exists(), f"Annotation file not found: {COCO_ANN_FILE}"

n_imgs = len(list(COCO_IMAGES_DIR.glob("*.jpg")))
print(f"COCO val2017 images found: {n_imgs}")
print(f"COCO annotation file found: {COCO_ANN_FILE}")

assert n_imgs > 0, "No COCO images found after extraction!"


## 2) Mini-YOLO architecture

Now we define our Mini-YOLO architecture like before - please refer to the last Jupyter notebook for further details!

In [ ]:
# MiniYOLO model definition
class MiniYOLO(nn.Module):
    def __init__(self, grid_size=7):
        super().__init__()
        self.S = grid_size
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 7, stride=2, padding=3), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 5, 1)
        )
    def forward(self, x):
        f = self.backbone(x)
        h = self.head(f)
        h = F.interpolate(h, size=(self.S, self.S), mode="bilinear", align_corners=False)
        return torch.sigmoid(h).permute(0,2,3,1)


We load the pretrained weights from our previous training on synthetic shapes:

In [ ]:
S = 7
IMG_SIZE = 128

model = MiniYOLO(S).to(device)
weights_path = "../../demos/mini_yolo_shapes.pt"
if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
    print("Loaded pretrained weights from", weights_path)
else:
    print("No pretrained weights found — training will start from scratch.")


## 3) Dataset and label parsing

We will train a person detector, so we train only one class. Since COCO contains many more classes, we need to prepare our dataset:

In [ ]:
# COCO -> Mini-YOLO label encoding (class-agnostic, but we train on a single class for stability)
# Here we focus on a single category ("person") to reduce COCO complexity and improve qualitative results.
CATEGORIES = ["person"]

# Filter out very small instances (COCO has many tiny persons that become a few pixels after resizing).
# COCO's ann["area"] is in original pixel units.
MIN_AREA = 32 * 32

def resize_boxes_to_target(boxes, orig_w, orig_h, target_size=IMG_SIZE):
    """Scale COCO boxes (x1,y1,x2,y2) to the resized image (IMG_SIZE x IMG_SIZE)."""
    if len(boxes) == 0:
        return np.zeros((0,4), dtype=np.float32)
    boxes = np.array(boxes, dtype=np.float32)
    sx = target_size / float(orig_w)
    sy = target_size / float(orig_h)
    boxes[:, [0,2]] *= sx  # x1, x2
    boxes[:, [1,3]] *= sy  # y1, y2
    return boxes

def encode_yolo_targets_from_boxes(boxes, grid_size=S, img_size=IMG_SIZE):
    """Encode multiple boxes into (S,S,5) with cell-relative offsets (one box per cell).

    Note: If multiple objects fall into the same cell, the last one wins.
    This keeps the implementation simple for teaching purposes.
    """
    y = torch.zeros((grid_size, grid_size, 5), dtype=torch.float32)
    if boxes.shape[0] == 0:
        return y
    x1, y1, x2, y2 = boxes[:,0], boxes[:,1], boxes[:,2], boxes[:,3]
    xc = ((x1 + x2) / 2) / img_size
    yc = ((y1 + y2) / 2) / img_size
    w  = (x2 - x1) / img_size
    h  = (y2 - y1) / img_size

    # Map to grid cells
    cell_x = np.clip((xc * grid_size).astype(int), 0, grid_size-1)
    cell_y = np.clip((yc * grid_size).astype(int), 0, grid_size-1)
    x_rel = xc * grid_size - cell_x
    y_rel = yc * grid_size - cell_y

    for cx, cy, xr, yr, wv, hv in zip(cell_x, cell_y, x_rel, y_rel, w, h):
        # Basic sanity: ignore degenerate boxes
        if wv <= 0 or hv <= 0:
            continue
        y[cy, cx] = torch.tensor([xr, yr, wv, hv, 1.0])
    return y

class CocoValDataset(Dataset):
    def __init__(
        self,
        coco_root=COCO_ROOT,
        n=500,
        categories=CATEGORIES,
        return_masks=False,
        min_area=MIN_AREA,
        require_category=True,
    ):
        """COCO val2017 dataset wrapper.

        Args:
            coco_root: folder containing val2017/ and annotations/
            n: number of images to sample (keep small for speed)
            categories: list of category names (e.g. ['person']) or None for all
            return_masks: if True, additionally returns a binary mask (IMG_SIZE x IMG_SIZE)
            min_area: filter instances with ann["area"] < min_area
            require_category: if True and categories!=None, only select images that contain that category.
                              This is recommended for stable single-class training.
        """
        self.images_dir = Path(coco_root) / "val2017"
        self.ann_file   = Path(coco_root) / "annotations" / "instances_val2017.json"
        self.return_masks = return_masks
        self.min_area = float(min_area) if min_area is not None else 0.0

        try:
            from pycocotools.coco import COCO
        except Exception as e:
            raise ImportError(
                "pycocotools is required for COCO. Install it via:\n"
                "  pip install pycocotools\n"
                "or (recommended on some systems):\n"
                "  conda install -c conda-forge pycocotools\n"
            ) from e

        self.coco = COCO(self.ann_file.as_posix())

        if categories is None:
            self.cat_ids = None
            img_ids = self.coco.getImgIds()
        else:
            cat_ids = []
            for name in categories:
                ids = self.coco.getCatIds(catNms=[name])
                if len(ids) == 0:
                    raise ValueError(f"COCO category '{name}' not found. Check spelling.")
                cat_ids.extend(ids)
            self.cat_ids = sorted(set(cat_ids))

            if require_category:
                # Only images that contain the selected category (best for single-class training stability)
                img_ids = self.coco.getImgIds(catIds=self.cat_ids)
            else:
                # All images, but we'll keep only the selected category's anns per image (useful for mixing negatives)
                img_ids = self.coco.getImgIds()

        img_ids = sorted(img_ids)
        if n is not None:
            img_ids = img_ids[:n]
        self.img_ids = img_ids

        assert len(self.img_ids) > 0, "No COCO images selected. Try categories=None, require_category=False, or increase n."

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs([img_id])[0]
        img_path = self.images_dir / img_info["file_name"]

        img_bgr = cv2.imread(img_path.as_posix())
        if img_bgr is None:
            img_bgr = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
            orig_h, orig_w = IMG_SIZE, IMG_SIZE
        else:
            orig_h, orig_w = img_bgr.shape[:2]

        # Load annotations for this image (filtered to selected categories)
        ann_ids = self.coco.getAnnIds(imgIds=[img_id], catIds=self.cat_ids, iscrowd=None)
        anns = self.coco.loadAnns(ann_ids)

        # Convert COCO bboxes: [x, y, w, h] -> [x1, y1, x2, y2]
        boxes = []
        valid_anns = []
        for ann in anns:
            if self.min_area and float(ann.get("area", 0.0)) < self.min_area:
                continue

            x, y, w, h = ann.get("bbox", [0,0,0,0])
            if w <= 1 or h <= 1:
                continue
            x1, y1, x2, y2 = x, y, x + w, y + h

            # Clip to image bounds
            x1 = max(0, min(x1, orig_w-1))
            y1 = max(0, min(y1, orig_h-1))
            x2 = max(0, min(x2, orig_w-1))
            y2 = max(0, min(y2, orig_h-1))
            if x2 <= x1 or y2 <= y1:
                continue

            boxes.append([x1, y1, x2, y2])
            valid_anns.append(ann)

        # Resize image and boxes
        img_bgr_rs = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        boxes_rs = resize_boxes_to_target(boxes, orig_w, orig_h, IMG_SIZE)
        y = encode_yolo_targets_from_boxes(boxes_rs, grid_size=S, img_size=IMG_SIZE)

        # Convert image to tensor (CHW, RGB, [0,1])
        img_rgb = cv2.cvtColor(img_bgr_rs, cv2.COLOR_BGR2RGB)
        img_t = torch.from_numpy(img_rgb.astype(np.float32)/255.).permute(2,0,1)

        if not self.return_masks:
            return img_t, y

        # Optional: build a binary union mask from the selected instances (for later segmentation work)
        mask = np.zeros((orig_h, orig_w), dtype=np.uint8)
        for ann in valid_anns:
            try:
                m = self.coco.annToMask(ann).astype(np.uint8)
                mask = np.maximum(mask, m)
            except Exception:
                pass
        mask_rs = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        mask_t = torch.from_numpy(mask_rs.astype(np.float32))  # (H,W), values 0/1

        return img_t, y, mask_t


Now let's create training and validation loaders. You can increase the data sizes `n` to get improved results.

In [ ]:
# Create datasets and dataloaders

train_ds = CocoValDataset(n=1500, categories=CATEGORIES, return_masks=False)
val_ds   = CocoValDataset(n=300,  categories=CATEGORIES, return_masks=False)

import os
BATCH = 32 if torch.cuda.is_available() else 8
NW = min(4, os.cpu_count() or 2)

train_loader = DataLoader(
    train_ds, batch_size=BATCH, shuffle=True,
    num_workers=NW, pin_memory=torch.cuda.is_available(),
    persistent_workers=(NW > 0)
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH, shuffle=False,
    num_workers=NW, pin_memory=torch.cuda.is_available(),
    persistent_workers=(NW > 0)
)

print(f"Train/Val sizes: {len(train_ds)} / {len(val_ds)} (change n=... to use more)")

## 4) YOLO-style loss

Here we define our YOLO loss. Note that we are using a different loss function now because the COCO dataset is far more complex than the synthetic shapes dataset. We still use an MSE-based YOLOv1-style loss, but we apply several modifications to make training more stable on real-world data.
The loss separates the coordinate terms into an $x$, $y$ loss and a $\sqrt w$, $\sqrt h$ loss, which is a classic YOLOv1 trick to reduce the impact of large bounding boxes.
Object cells are weighted more strongly than non-object cells, and the entire loss is additionally scaled to stabilize gradients.
Although this formulation differs from the simpler loss used for synthetic shapes, it is still a regression-based YOLOv1 loss.

Let each prediction be 
$(\hat{x}_i, \hat{y}_i, \hat{w}_i, \hat{h}_i, \hat{c}_i)$
and the corresponding ground truth
$(x_i, y_i, w_i, h_i, c_i)$,
with $c_i \in \{0,1\}$ indicating object presence.
The total loss consists of coordinate loss, size loss,
objectness loss, and a penalty for non-object cells:

$$
\mathcal{L}
= 0.1 \Bigg(
\lambda_{\mathrm{coord}} \, \mathcal{L}_{xy}
+ \lambda_{\mathrm{coord}} \, \mathcal{L}_{wh}
+ 2 \, \mathcal{L}_{\mathrm{obj}}
+ \lambda_{\mathrm{noobj}} \, \mathcal{L}_{\mathrm{noobj}}
\Bigg).
$$

Let's take a closer look at the components. The **Coordinate Loss** for $x$ and $y$ uses Mean Squared Error:
$$
\mathcal{L}_{xy}
=
\frac{1}{N_{\mathrm{obj}}}
\sum_{i \in \mathrm{obj}}
\left[
(\hat{x}_i - x_i)^2
+
(\hat{y}_i - y_i)^2
\right].
$$

Width and height losses are calculated similarly, but with a square root to reduce the impact of large bounding boxes:
$$
\mathcal{L}_{wh}
=
\frac{1}{N_{\mathrm{obj}}}
\sum_{i \in \mathrm{obj}}
\left[
\left(\sqrt{\hat{w}_i} - \sqrt{w_i}\right)^2
+
\left(\sqrt{\hat{h}_i} - \sqrt{h_i}\right)^2
\right].
$$

As before, the **Objectness Loss** and **No-Object Loss** are defined as
$$
\mathcal{L}_{\mathrm{obj}}
=
\frac{1}{N_{\mathrm{obj}}}
\sum_{i \in \mathrm{obj}}
(\hat{c}_i - 1)^2
$$

and
$$
\mathcal{L}_{\mathrm{noobj}}
=
\frac{1}{N_{\mathrm{noobj}}}
\sum_{i \in \mathrm{noobj}}
(\hat{c}_i - 0)^2 .
$$


In [ ]:
# Define YOLO loss function
def yolo_loss(pred, target, lambda_coord=3.0, lambda_noobj=2.0):
    obj_mask = target[..., 4] > 0.5 # object present
    noobj_mask = ~obj_mask # no object present

    if obj_mask.any():
        # Compute coordinate losses only for cells with objects
        xy_loss = lambda_coord * F.mse_loss(pred[obj_mask][..., :2], target[obj_mask][..., :2])
        wh_loss = lambda_coord * F.mse_loss(
            torch.sqrt(pred[obj_mask][..., 2:4].clamp(1e-6, 1.0)),
            torch.sqrt(target[obj_mask][..., 2:4].clamp(1e-6, 1.0))
        )
        # Objectness loss for cells with objects, weighted higher
        obj_loss = 2.0 * F.mse_loss(pred[obj_mask][..., 4], target[obj_mask][..., 4])
    else:
        xy_loss = wh_loss = obj_loss = torch.tensor(0., device=pred.device)

    # Objectness loss for cells without objects, weighted lower
    noobj_loss = lambda_noobj * F.mse_loss(pred[noobj_mask][..., 4], target[noobj_mask][..., 4])
    
    # Return scaled total loss
    return 0.1 * (xy_loss + wh_loss + obj_loss + noobj_loss)


## 5) Fine-tuning

**TODO:** Now it's your turn! Define an optimizer - remember that for fine-tuning, we use a rather low learning rate. Then, train the model and print the validation loss every epoch.

In [ ]:
# YOUR CODE HERE


## 6) Postprocessing and Evaluation

Finally, we apply Non-Maximum Suppression (NMS) to the raw model predictions to filter out duplicate detections and visualize the results qualitatively.

In [ ]:
import torchvision.ops as ops

def yolo_postprocess(pred, conf_thresh=0.30, iou_thresh=0.40):
    """Convert (S,S,5) to pixel boxes with NMS.

    Tips:
      - If you see too many random boxes, increase conf_thresh (e.g. 0.4–0.5).
      - If boxes duplicate, lower iou_thresh slightly (e.g. 0.3).
    """
    S_ = pred.shape[0]
    boxes, scores = [], []

    # Convert cell-relative predictions to global image coordinates
    for cy in range(S_):
        for cx in range(S_):
            x_rel, y_rel, w, h, conf = pred[cy, cx]
            if conf < conf_thresh:
                continue

            # Convert to global normalized coordinates
            x = (cx + x_rel) / S_
            y = (cy + y_rel) / S_

            # Convert to absolute pixel coordinates
            x1 = (x - w / 2) * IMG_SIZE
            y1 = (y - h / 2) * IMG_SIZE
            x2 = (x + w / 2) * IMG_SIZE
            y2 = (y + h / 2) * IMG_SIZE

            boxes.append([x1, y1, x2, y2])
            scores.append(conf)

    if len(boxes) == 0:
        return []

    boxes = torch.tensor(boxes, dtype=torch.float32)
    scores = torch.tensor(scores, dtype=torch.float32)

    # Perform NMS
    keep = ops.nms(boxes, scores, iou_thresh)
    boxes = boxes[keep]
    scores = scores[keep]
    out = []
    for b, s in zip(boxes, scores):
        out.append((b.tolist(), float(s)))
    return out


In [ ]:
# Quick qualitative check: visualize predictions vs. ground truth on a few validation images

def yolo_decode_gt(target):
    """Decode GT (S,S,5) into pixel boxes (without NMS)."""
    boxes = []
    for cy in range(S):
        for cx in range(S):
            xr, yr, w, h, conf = target[cy, cx].tolist()
            if conf < 0.5:
                continue
            x = (cx + xr) / S
            y = (cy + yr) / S
            x1 = (x - w/2) * IMG_SIZE
            y1 = (y - h/2) * IMG_SIZE
            x2 = (x + w/2) * IMG_SIZE
            y2 = (y + h/2) * IMG_SIZE
            boxes.append([x1,y1,x2,y2])
    return boxes

def draw_boxes(ax, boxes, color, linewidth=2):
    for x1,y1,x2,y2 in boxes:
        ax.add_patch(plt.Rectangle((x1,y1), x2-x1, y2-y1, fill=False, linewidth=linewidth, edgecolor=color))

model.eval()
n_show = 6
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

with torch.no_grad():
    for i in range(n_show):
        img_t, y = val_ds[i]
        p = model(img_t.unsqueeze(0).to(device)).squeeze(0).cpu()

        pred = yolo_postprocess(p, conf_thresh=0.70, iou_thresh=0.40)
        pred_boxes = [b for (b, s) in pred]
        gt_boxes = yolo_decode_gt(y)

        img_np = (img_t.permute(1,2,0).numpy() * 255).astype('uint8')

        ax = axes[i]
        ax.imshow(img_np)
        draw_boxes(ax, gt_boxes, color='lime', linewidth=2)     # GT
        draw_boxes(ax, pred_boxes, color='red', linewidth=2)    # Pred
        ax.set_title(f"GT (green) vs Pred (red): {len(gt_boxes)} / {len(pred_boxes)}")
        ax.axis('off')

plt.tight_layout()
plt.show()
